# Activity #2 — SOLUTION (Instructor Reference)

**Answers:**

| File | Sensor | Attack |
|---|---|---|
| `attack_A.csv` | s11 (Ps30 — static pressure HPC outlet) | constant drift, +1.5σ |
| `attack_B.csv` | s14 (NRc — corrected core speed) | stuck-at value (frozen at cycle 30) |
| `attack_C.csv` | s9 (Nc — physical core speed) | gaussian noise, σ = 0.3 × channel std |

**Why these sensors:** all three are among the most predictive in CMAPSS FD001 — a real adversary would target sensors the model leans on, not random ones. This forces students to actually use the model explanations rather than just eyeballing.

**Pedagogical point:**
- The naive approach (Tool A — eyeball every channel) works but is slow.
- The quantitative approach (Tool B — rank divergence) is fast but tells you nothing about *whether the model cares*.
- The explanation approach (Tool C — soft tree split weights) directly shows which sensor the model's decision depends on. This is the only approach that scales to high-dimensional / production systems.


In [ ]:
!pip install -q neural-trees pandas matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from neural_trees import SoftDecisionTree

In [ ]:
BASE = 'https://raw.githubusercontent.com/cgrtml/wsu-workshop-may15/main/data'
for f in ['train_FD001.txt', 'engine17_clean.csv', 'attack_A.csv', 'attack_B.csv', 'attack_C.csv']:
    !wget -q -nc {BASE}/{f}

In [ ]:
cols = ['engine_id', 'cycle'] + [f'op{i}' for i in range(1, 4)] + [f's{i}' for i in range(1, 22)]
train = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None, names=cols)
max_cycles = train.groupby('engine_id')['cycle'].max().rename('max_cycle')
train = train.merge(max_cycles, on='engine_id')
train['RUL'] = (train['max_cycle'] - train['cycle']).clip(upper=125)

def bin_rul(r):
    if r < 30: return 0
    if r < 80: return 1
    return 2

train['health'] = train['RUL'].apply(bin_rul)
informative = [f's{i}' for i in range(1, 22) if train[f's{i}'].std() > 0.001]
scaler = StandardScaler().fit(train[informative].values)
X = scaler.transform(train[informative].values); y = train['health'].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

tree = SoftDecisionTree(depth=4, max_epochs=30, learning_rate=0.01, penalty_coef=1e-3, verbose=False).fit(X_tr, y_tr)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
labels = ['Critical', 'Caution', 'Healthy']

In [ ]:
# Quick-and-dirty divergence ranking — confirms expected answers
def rank_divergence(attack_file):
    clean = pd.read_csv('engine17_clean.csv')
    attack = pd.read_csv(attack_file)
    diffs = {s: float(np.mean(np.abs(clean[s].values - attack[s].values))) for s in informative}
    return sorted(diffs.items(), key=lambda kv: -kv[1])

for f in ['attack_A.csv', 'attack_B.csv', 'attack_C.csv']:
    print(f'--- {f} ---  Top sensor: {rank_divergence(f)[0]}')

In [ ]:
# Side-by-side: clean vs attack for the manipulated sensor in each file
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (f, sensor) in zip(axes, [('attack_A.csv', 's11'), ('attack_B.csv', 's14'), ('attack_C.csv', 's9')]):
    clean = pd.read_csv('engine17_clean.csv')
    attack = pd.read_csv(f)
    ax.plot(clean['cycle'], clean[sensor], label='clean', linewidth=2)
    ax.plot(attack['cycle'], attack[sensor], label='attack', alpha=0.8)
    ax.set_title(f'{f} — {sensor}')
    ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Show that random forest changes predictions but cannot localize the cause
def predict_engine(file_path, model):
    df = pd.read_csv(file_path)
    Xq = scaler.transform(df[informative].values)
    return model.predict(Xq)

clean_tree_pred = predict_engine('engine17_clean.csv', tree)
clean_rf_pred = predict_engine('engine17_clean.csv', rf)

for f in ['attack_A.csv', 'attack_B.csv', 'attack_C.csv']:
    p_tree = predict_engine(f, tree)
    p_rf = predict_engine(f, rf)
    print(f'{f}: SoftTree changed {(p_tree != clean_tree_pred).sum():3d} cycles · '
          f'RandomForest changed {(p_rf != clean_rf_pred).sum():3d} cycles')